# Finance-Specific Synthetic Dataset Version

In [ ]:
import pandas as pd
import numpy as np

# Load finance-specific synthetic dataset
df_fin = pd.read_csv("financial_fraud_synthetic.csv")

df_fin['timestamp'] = pd.to_datetime(df_fin['timestamp'])

# Sort by account and time
df_fin = df_fin.sort_values(['account_id', 'timestamp']).reset_index(drop=True)

# Time gap feature
df_fin['prev_timestamp'] = df_fin.groupby('account_id')['timestamp'].shift(1)
df_fin['time_gap_minutes'] = (df_fin['timestamp'] - df_fin['prev_timestamp']).dt.total_seconds() / 60
df_fin['time_gap_minutes'] = df_fin['time_gap_minutes'].fillna(1440)

# Account transaction count
df_fin['account_txn_count'] = df_fin.groupby('account_id').cumcount() + 1

# Account average amount and deviation
df_fin['account_avg_amount'] = df_fin.groupby('account_id')['amount'].transform('mean')
df_fin['amount_deviation'] = df_fin['amount'] - df_fin['account_avg_amount']

# Finance-specific feature set for Mahalanobis
features_fin = [
    'amount',
    'time_gap_minutes',
    'account_txn_count',
    'amount_deviation',
    'is_international',
    'failed_login_count',
    'device_change_flag',
    'ip_risk_score',
    'amount_to_balance_ratio',
    'new_beneficiary_flag'
]

X_fin = df_fin[features_fin]

# Mean vector and covariance matrix
mu_fin = X_fin.mean()
Sigma_fin = X_fin.cov()

mu_fin_np = mu_fin.values
Sigma_fin_np = Sigma_fin.values
Sigma_fin_inv = np.linalg.inv(Sigma_fin_np)

# Mahalanobis scores
X_fin_np = X_fin.values
mahalanobis_scores_fin = []

for i in range(len(X_fin_np)):
    diff = X_fin_np[i] - mu_fin_np
    score = np.sqrt(diff.T @ Sigma_fin_inv @ diff)
    mahalanobis_scores_fin.append(score)

df_fin['mahalanobis_score'] = mahalanobis_scores_fin

# EWMA on amount per account
df_fin['ewma_amount'] = df_fin.groupby('account_id')['amount'].transform(
    lambda x: x.ewm(alpha=0.3).mean()
)

df_fin['ewma_deviation'] = abs(df_fin['amount'] - df_fin['ewma_amount'])

# Normalize scores
df_fin['mah_norm'] = df_fin['mahalanobis_score'] / df_fin['mahalanobis_score'].max()
df_fin['ewma_norm'] = df_fin['ewma_deviation'] / df_fin['ewma_deviation'].max()

# Final combined risk score
df_fin['final_risk_score'] = 0.6 * df_fin['mah_norm'] + 0.4 * df_fin['ewma_norm']

# Risk bands
df_fin['risk_level'] = pd.cut(
    df_fin['final_risk_score'],
    bins=[0, 0.4, 0.7, 1],
    labels=['Low', 'Medium', 'High']
)

# Show result preview
df_fin[['account_id', 'amount', 'mahalanobis_score', 'ewma_deviation', 'final_risk_score', 'risk_level']].head(10)

In [ ]:
import matplotlib.pyplot as plt

account_choice = "A1046"

acc_df = df_fin[df_fin['account_id'] == account_choice].copy()

print("Account selected:", account_choice)
print("Number of transactions:", len(acc_df))
print("Average risk score:", acc_df['final_risk_score'].mean())
print("Maximum risk score:", acc_df['final_risk_score'].max())
print("Average Mahalanobis score:", acc_df['mahalanobis_score'].mean())
print("Maximum Mahalanobis score:", acc_df['mahalanobis_score'].max())

acc_df[[
    'account_id', 'timestamp', 'amount', 'is_international',
    'failed_login_count', 'device_change_flag',
    'new_beneficiary_flag', 'mahalanobis_score',
    'ewma_deviation', 'final_risk_score', 'risk_level'
]].sort_values('timestamp')

In [ ]:
plt.figure()
plt.plot(acc_df['timestamp'], acc_df['amount'], label='Actual Amount')
plt.plot(acc_df['timestamp'], acc_df['ewma_amount'], label='EWMA Trend')
plt.title(f"Amount vs EWMA for Account {account_choice}")
plt.xlabel("Time")
plt.ylabel("Transaction Amount")
plt.legend()
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure()
plt.plot(acc_df['timestamp'], acc_df['mahalanobis_score'], marker='o')
plt.title(f"Mahalanobis Score Over Time for Account {account_choice}")
plt.xlabel("Time")
plt.ylabel("Mahalanobis Score")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure()
plt.hist(acc_df['mahalanobis_score'], bins=10)
plt.title(f"Mahalanobis Score Distribution for Account {account_choice}")
plt.xlabel("Mahalanobis Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df_fin['risk_level'].value_counts()

In [ ]:
df_fin.sort_values('final_risk_score', ascending=False)[
    ['account_id', 'amount', 'is_international', 'failed_login_count',
     'device_change_flag', 'new_beneficiary_flag',
     'final_risk_score', 'risk_level']
].head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(df_fin['mahalanobis_score'], bins=50)
plt.title("Mahalanobis Score Distribution")
plt.xlabel("Mahalanobis Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df_fin['risk_level'].value_counts().plot(kind='bar')
plt.title("Risk Level Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Number of Transactions")
plt.show()

In [ ]:
# Create binary prediction from risk level
df_fin['predicted_fraud'] = (df_fin['risk_level'] == 'High').astype(int)

# Confusion matrix values
TP = ((df_fin['fraud_label'] == 1) & (df_fin['predicted_fraud'] == 1)).sum()
TN = ((df_fin['fraud_label'] == 0) & (df_fin['predicted_fraud'] == 0)).sum()
FP = ((df_fin['fraud_label'] == 0) & (df_fin['predicted_fraud'] == 1)).sum()
FN = ((df_fin['fraud_label'] == 1) & (df_fin['predicted_fraud'] == 0)).sum()

# Metrics
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (TP + TN) / len(df_fin)

# Print results
print("Confusion Matrix")
print("----------------")
print("TP:", TP)
print("FP:", FP)
print("TN:", TN)
print("FN:", FN)

print("\nEvaluation Metrics")
print("------------------")
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)
print("Accuracy:", accuracy)

In [ ]:
import pandas as pd

conf_matrix = pd.DataFrame(
    [[TN, FP],
     [FN, TP]],
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1']
)

conf_matrix